# Phase 1 — Offline baseline

Visual validation of the generator, then a LightGBM baseline.

**Order matters.** Do not skip to training. If the histograms in Step 2 look wrong,
the model in Step 5 is measuring a bug, not fraud. Fix the generator first.

Code cells are `TODO(you)` skeletons — see `docs/PHASE_1.md`, Steps 4-6.

## Step 0 — Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make `src` importable when the kernel starts in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

DATA = ROOT / "data" / "transactions.parquet"
SEED = 42

## Step 1 — Load data

Generate it first if the file is missing:

```bash
python -m src.generator.generate --n-users 1000 --n-transactions 500000 \
    --fraud-rate 0.003 --seed 42 --output data/transactions.parquet
```

In [ ]:
# TODO(you):
#   df = pd.read_parquet(DATA)
#   df = df.sort_values("timestamp", ignore_index=True)
# Then check: shape, dtypes, timestamp min/max, df["is_fraud"].mean().
# The observed fraud rate should be close to the --fraud-rate you asked for.
df = None

## Step 2 — Sanity-check the generator

Three plots. Each one has a specific failure it is designed to catch.

### 2a — Amount distribution

Expect a right-skewed log-normal: most purchases small, a thin long tail.
Plot `log10(amount)` too — it should look roughly bell-shaped.

**Catches:** amounts sampled from the wrong distribution; a spike of tiny amounts
leaking out of the normal layer (card-testing probes should only exist in fraud rows).

In [ ]:
# TODO(you): histogram amount for is_fraud==0 vs is_fraud==1, overlaid, log-x.
# Fraud should sit visibly to the right (amount anomalies) AND have a small cluster
# far to the left (card-testing probes).

### 2b — Hour of day

Expect a clear daytime bulge and a quiet 3-5 a.m. trough.

**Catches:** a flat histogram means timestamps ignored `active_hours`; a perfectly
empty night means you gave `active_hours` no escape hatch, and the model will just
memorise the rule.

In [ ]:
# TODO(you):
#   df["hour"] = df["timestamp"].dt.hour
#   then value_counts().sort_index().plot.bar(), split by is_fraud

### 2c — Distance from home

Needs the user profiles, which are NOT in the parquet — regenerate them with the same
seed (`generate_user_profiles(n_users, seed=SEED)`) and join on `user_id`.

Expect most transactions within a few km of home, a modest travel tail, and fraud
reaching far beyond both.

**Catches:** geo jitter in the wrong units (degrees vs km); a fraud tail that overlaps
the normal travel tail, which would make geo-velocity unlearnable.

In [ ]:
# TODO(you): implement haversine(lat1, lon1, lat2, lon2) -> km, vectorised over numpy
# arrays. You will need this again for the velocity features in Step 3 and again in
# Phase 2 — write it once, somewhere importable (src/features/), not inline here.
#
#   from src.generator.profile import generate_user_profiles
#   profiles = generate_user_profiles(n_users=..., seed=SEED)
#   home = pd.DataFrame([{...} for p in profiles]).set_index("user_id")
#   df = df.join(home, on="user_id")
#   df["dist_from_home_km"] = haversine(...)

## Step 3 — Feature engineering (placeholder)

Per-card velocity features, computed from **prior rows only**.

> **Look-ahead bias is the one mistake that silently ruins this phase.** Any
> `groupby.transform("mean")`, any centred rolling window, any `.shift(-1)` uses the
> future. Sort by timestamp, group by `card_id`, and only ever look backwards.

Target feature set (docs/PHASE_1.md, Step 4):

| Feature | Catches |
|---|---|
| count in last 1 min / 10 min / 1 h for this card | burst |
| km since previous txn ÷ hours since previous txn | geo-velocity |
| amount z-score vs the card's *running* mean/std | amount anomaly |
| seconds since previous txn | burst, card testing |
| is this merchant new for this card | all |

Write this so the same logic can be mirrored in the Phase 2 streaming engine —
i.e. as functions over a card's past, not as whole-DataFrame operations.

In [ ]:
# TODO(you): build the feature frame.
# Hints:
#   - time-window counts: set a DatetimeIndex, then
#     df.groupby("card_id").rolling("10min").count() — rolling on a time offset is
#     backward-looking by default, which is what you want. Verify that on a toy frame
#     before trusting it on 500k rows.
#   - running mean/std: groupby("card_id")["amount"].expanding().mean().shift(1)
#     The .shift(1) is what excludes the current row from its own baseline. Without
#     it you have leaked the label into the feature.
#   - previous location/time: groupby("card_id")[[...]].shift(1)
#   - new merchant: groupby(["card_id", "merchant_id"]).cumcount() == 0
#
# Then assert no feature was built from a future row — spot-check one card by hand.
FEATURES: list[str] = []

## Step 4 — Temporal split

Train on earlier data, test on later. **Never** `train_test_split(shuffle=True)` —
a random split puts a card's future in train and its past in test, and your PR-AUC
becomes fiction.

In [ ]:
# TODO(you):
#   cutoff = df["timestamp"].quantile(0.8)
#   train = df[df["timestamp"] <= cutoff]
#   test  = df[df["timestamp"] >  cutoff]
# Check the fraud rate in each split — if test has ~0 positives the metrics below are
# meaningless and you need a longer window or a higher fraud rate.

## Step 5 — Train LightGBM

Baseline only. No deep learning, no hyperparameter search — save the energy for
Phase 2's streaming system.

In [ ]:
# TODO(you):
#   import lightgbm as lgb
#   Handle imbalance with scale_pos_weight = (n_negative / n_positive) computed on
#   TRAIN only. Start there before touching SMOTE or any resampling.
#   Sensible starting params: objective="binary", n_estimators=500, learning_rate=0.05,
#   num_leaves=31, random_state=SEED.
#   Use an early-stopping set carved from the END of train (temporally), not a random
#   slice — same leakage rule as Step 4.

## Step 6 — Evaluate honestly

Report **PR-AUC**, **precision**, **recall**, **confusion matrix**.

Do not report accuracy. At 0.3% fraud, predicting "never fraud" scores 99.7% and
catches nothing.

In [ ]:
# TODO(you):
#   from sklearn.metrics import (average_precision_score, confusion_matrix,
#                                precision_recall_curve, classification_report)
#   - average_precision_score(y_test, y_prob) is your headline PR-AUC
#   - plot the full precision-recall curve; a single threshold hides the trade-off
#   - pick the operating threshold from that curve based on how many false positives
#     an analyst could actually review per day, not from the default 0.5

## Step 7 — Interrogate the model

Two checks before declaring Phase 1 done:

1. **Feature importance** — the velocity features should rank high. If raw `amount`
   or `hour` dominates, Step 3 is wrong or the fraud patterns are too easy.
2. **Eyeball the flagged rows** — pull the highest-scoring test transactions and read
   them. They should look suspicious to a human. Then pull the false negatives and
   work out which pattern is being missed.

In [ ]:
# TODO(you): plot lgb.plot_importance(model, importance_type="gain"), then inspect
# the top-scoring and the missed transactions by hand.
#
# If a feature is suspiciously dominant, suspect leakage before celebrating.